In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score
import warnings
warnings.filterwarnings('ignore')   

In [4]:
df =pd.read_csv(r"D:\DA\Archive\GitHUB\Forage-QA\Creditriskdata.csv")
df.head(10)

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0
5,4661159,0,5376.886873,7189.121298,85529.84591,2,697,0
6,8291909,1,3634.057471,7085.980095,68691.57707,6,722,0
7,4616950,4,3302.172238,13067.570210,50352.16821,3,545,1
8,3395789,0,2938.325123,1918.404472,53497.37754,4,676,0
9,4045948,0,5396.366774,5298.824524,92349.55399,2,447,0


In [7]:
df.shape

(10000, 8)

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

x=df.drop('default',axis=1)
y=df['default']

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42)

scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

model= LogisticRegression()
model.fit(x_train_scaled,y_train)

prob= model.predict_proba(x_test_scaled)[:,1]
print(roc_auc_score(y_test,prob))


0.9999512955386713


Here the score is nearly perfect , there may be a chance of data leaking of copy patterns in your test data 

In [10]:
from sklearn.tree import DecisionTreeClassifier

model2= DecisionTreeClassifier()
model2.fit(x_train_scaled,y_train)

model2_predict=model2.predict_proba(x_test_scaled)[:,1]
print(roc_auc_score(y_test,model2_predict))

0.9887318749826056


In [19]:
def expected_loss(model,scaler,income,loan_amount,credit_score,age,recovery_rate=0.1):
    data = pd.DataFrame([{
        'customer_id': 0,
        'credit_lines_outstanding': 0,
        'loan_amt_outstanding': loan_amount,
        'total_debt_outstanding': loan_amount,
        'income': income,
        'years_employed': age,
        'fico_score': credit_score
    }])
    scaled_data = scaler.transform(data)
    pd_value = model.predict_proba(scaled_data)[0][1]
    loss = loan_amount * pd_value * (1 - recovery_rate)
    return loss

result = expected_loss(
    model=model,
    scaler=scaler,
    income=50000,
    loan_amount=200000,
    credit_score=650,
    age=35
)

print(result)


180000.0
